### What is ReAct?
ReAct (Reasoning + Acting) is a framework where an LLM:

- Reasons step-by-step (e.g. decomposes questions, makes decisions)

- Acts by calling tools like search, calculators, or retrievers

This makes it perfect for Agentic RAG:
✅ Think → Retrieve → Observe → Reflect → Final Answer

In [1]:
import os
from langchain.agents import create_agent
from langchain_core.tools import Tool
from langchain_community.tools.wikipedia.tool import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import END
from langgraph.graph import StateGraph
from typing import Annotated,TypedDict,Sequence
from langchain_core.messages import BaseMessage,HumanMessage,AIMessage
from langgraph.graph.message import add_messages

d:\project\ragvenv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.1)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(
d:\project\ragvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
#create retriever tool
urls=[
    "https://vnrvjiet.ac.in/",
    "https://vnrvjiet.ac.in/admission/",
    "https://vnrvjiet.ac.in/it/",
    "https://vnrvjiet.ac.in/assets/pdfs/IPITEx%202024%20awards.pdf"
]

docs = [doc for url in urls for doc in WebBaseLoader(url).load()]

In [3]:
splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50)
chunks=splitter.split_documents(docs)

embeddings=HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")
persist_directory="./chroma_db"
vectorstore=Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=persist_directory, #directory to store the data
    collection_name="rag_collection"
)
retriever=vectorstore.as_retriever()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4362.38it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
retriever.invoke("What are autonomous agents?")
query="Who is Mangathayaru?"
similar_docs=vectorstore.similarity_search(query,k=3)
similar_docs

[Document(metadata={'title': 'VNRVJIET', 'source': 'https://vnrvjiet.ac.in/it/', 'language': 'en'}, page_content='2025-2026\n\n\nPhoto Gallery\r\n                \n\n\n\n\nDr. Nimmala Mangathayaru has been honoured with the prestigious Bharat Education Excellence Award as Jyestha Acharya in recognition of her exemplary contributions to the field of education and academic leadership. \r\n                  \n\n\n2024-2025\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nNews letter 2023-24\r\n              \n\n\nNews letter 2022-23\r\n              \n\n\n\n\n\n\n\n\nSWOC\n\n\n\r\n          SWOC June 2024'),
 Document(metadata={'source': 'https://vnrvjiet.ac.in/it/', 'title': 'VNRVJIET', 'language': 'en'}, page_content='Dr N Mangathayaru\n\nithead@vnrvjiet.in\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nAbout\n\n\n\n\nSWOC\n\n\n\n\nAcademic Programs\n\n\n\n\n\r\n            Faculty and Staff\r\n          \n\